# Final NEDI x2 Evaluation — AWS Multi-CPU

Run this notebook from the repository root on the AWS server. It uses four worker processes to evaluate Set5, Set14, BSD100, and Urban100 at x2. Bicubic is rerun on the same AWS CPU for a fair timing comparison. Results are checkpointed after every image and can be resumed. The notebook still works in Colab as a fallback.

In [5]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print('AWS/local Jupyter environment detected.')


AWS/local Jupyter environment detected.


In [6]:
import os
import subprocess
from pathlib import Path

# Prevent each worker from secretly creating more numerical-library threads.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'

if IN_COLAB:
    REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    # Jupyter commonly starts in notebooks/, rather than the repository root.
    start_dir = Path.cwd().resolve()
    REPO_ROOT = next((directory for directory in (start_dir, *start_dir.parents)
                      if (directory / 'app').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find the repository above {start_dir}. ' 
            'Open this notebook from the repository or set its working directory to it.'
        )

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')


Repository ready: /home/ubuntu/Code/SuperResolution-Comparative-Analysis


In [7]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.data_validation import validate_prepared_dataset
from app.evaluation.bicubic import BicubicEvaluationConfig, evaluate_bicubic_image
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import pair_image_paths
from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_image
from app.evaluation.reporting import summarize_results

print('Bicubic and NEDI x2 evaluators imported successfully.')


Bicubic and NEDI x2 evaluators imported successfully.


In [8]:
from datetime import UTC, datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
import csv

default_data_root = '/content/drive/MyDrive/FYP_SR_Data' if IN_COLAB else '/mnt/fyp-data/FYP_SR_Data'
DATA_ROOT = Path(os.environ.get('FYP_SR_DATA_ROOT', default_data_root))
WORKERS = int(os.environ.get('FYP_NEDI_WORKERS', '4'))
COMPUTE_INSTANCE = os.environ.get('FYP_COMPUTE_INSTANCE', 'm7i.2xlarge')
# Leave as None for a new run. To resume after an interruption, replace
# None with the timestamped folder name printed by the earlier run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'x2_full' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')

if WORKERS < 1:
    raise ValueError('FYP_NEDI_WORKERS must be at least 1.')
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

print(f'Full NEDI x2 run: {RUN_ROOT}')
print(f'Compute: {COMPUTE_INSTANCE}; workers: {WORKERS}')
print('Protocol: 3 warm-ups and 10 timed CPU runs per image.')


Full NEDI x2 run: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x2_full/20260829_185005_utc
Compute: m7i.2xlarge; workers: 4
Protocol: 3 warm-ups and 10 timed CPU runs per image.


In [9]:
validations = {}
for dataset in DATASETS:
    validation = validate_prepared_dataset(dataset, 2, DATA_ROOT)
    validations[dataset] = validation
    print(f'VALID: {dataset} x2 has {validation.image_count} complete HR/LR pairs.')

print('All x2 datasets passed validation.')


VALID: Set5 x2 has 5 complete HR/LR pairs.
VALID: Set14 x2 has 14 complete HR/LR pairs.
VALID: BSD100 x2 has 100 complete HR/LR pairs.
VALID: Urban100 x2 has 100 complete HR/LR pairs.
All x2 datasets passed validation.


In [ ]:
def load_checkpoint(path):
    if not path.exists():
        return []
    with path.open(newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))

def run_dataset_method(dataset, method):
    validation = validations[dataset]
    pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
    checkpoint_csv = METRICS_ROOT / f'{dataset}_x2_{method}_aws_final.csv'
    records = load_checkpoint(checkpoint_csv)
    completed_images = {record['image'] for record in records}
    pending_pairs = [(hr, lr) for hr, lr in pairs if hr.name not in completed_images]

    if method == 'bicubic':
        config = BicubicEvaluationConfig(dataset=dataset, scale=2, warmup_runs=3, timed_runs=10)
        evaluator = evaluate_bicubic_image
    else:
        config = NEDIEvaluationConfig(
            dataset=dataset, scale=2, window_size=8, edge_threshold=8.0,
            warmup_runs=3, timed_runs=10,
        )
        evaluator = evaluate_nedi_image

    print(f'{dataset} x2 {method}: {len(completed_images)}/{len(pairs)} already complete.')
    with ProcessPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(evaluator, hr, lr, config): hr.name for hr, lr in pending_pairs}
        for future in as_completed(futures):
            record = future.result()
            record['compute_instance'] = COMPUTE_INSTANCE
            record['execution_mode'] = 'parallel_process_workers'
            record['worker_count'] = WORKERS
            records.append(record)
            records.sort(key=lambda item: item['image'])
            write_results_csv(records, checkpoint_csv, overwrite=True)
            print(
                f'{dataset} x2 {method}: {len(records)}/{len(pairs)} — {record["image"]} — '
                f'PSNR-Y={float(record["psnr_y"]):.4f}, '
                f'time={float(record["latency_mean_ms"]) / 1000:.2f}s'
            )
    return records

bicubic_records = []
nedi_records = []
for dataset in DATASETS:
    bicubic_records.extend(run_dataset_method(dataset, 'bicubic'))
    nedi_records.extend(run_dataset_method(dataset, 'nedi'))

print(f'Completed {len(nedi_records)} NEDI and {len(bicubic_records)} bicubic x2 evaluations.')


Set5 x2 bicubic: 0/5 already complete.
Set5 x2 bicubic: 1/5 — butterfly.png — PSNR-Y=27.4370, time=0.00s
Set5 x2 bicubic: 2/5 — bird.png — PSNR-Y=36.8216, time=0.00s
Set5 x2 bicubic: 3/5 — head.png — PSNR-Y=34.8814, time=0.00s
Set5 x2 bicubic: 4/5 — woman.png — PSNR-Y=32.1487, time=0.00s
Set5 x2 bicubic: 5/5 — baby.png — PSNR-Y=37.0778, time=0.00s
Set5 x2 nedi: 0/5 already complete.
Set5 x2 nedi: 1/5 — butterfly.png — PSNR-Y=27.1875, time=3.40s
Set5 x2 nedi: 2/5 — head.png — PSNR-Y=34.5194, time=4.02s
Set5 x2 nedi: 3/5 — bird.png — PSNR-Y=36.1426, time=4.43s
Set5 x2 nedi: 4/5 — woman.png — PSNR-Y=31.8153, time=4.11s
Set5 x2 nedi: 5/5 — baby.png — PSNR-Y=36.1171, time=13.38s
Set14 x2 bicubic: 0/14 already complete.
Set14 x2 bicubic: 1/14 — coastguard.png — PSNR-Y=29.1209, time=0.00s
Set14 x2 bicubic: 2/14 — comic.png — PSNR-Y=26.0060, time=0.00s
Set14 x2 bicubic: 3/14 — baboon.png — PSNR-Y=24.8617, time=0.00s
Set14 x2 bicubic: 4/14 — face.png — PSNR-Y=34.8539, time=0.00s
Set14 x2 bicubi

In [ ]:
nedi_combined_csv = METRICS_ROOT / 'nedi_x2_all_images_final.csv'
bicubic_combined_csv = METRICS_ROOT / 'bicubic_x2_aws_all_images_final.csv'
summary_csv = METRICS_ROOT / 'nedi_x2_summary_final.csv'
comparison_summary_csv = METRICS_ROOT / 'x2_aws_comparison_summary.csv'
summary_records = summarize_results(nedi_records)
comparison_summary = summarize_results(bicubic_records + nedi_records)
write_results_csv(nedi_records, nedi_combined_csv, overwrite=True)
write_results_csv(bicubic_records, bicubic_combined_csv, overwrite=True)
write_results_csv(summary_records, summary_csv, overwrite=True)
write_results_csv(comparison_summary, comparison_summary_csv, overwrite=True)

for row in summary_records:
    print(
        f"{row['dataset']} x2: images={row['image_count']}, "
        f"PSNR-Y={row['psnr_y']:.4f}, SSIM-Y={row['ssim_y']:.4f}, "
        f"PSNR-RGB={row['psnr_rgb']:.4f}, SSIM-RGB={row['ssim_rgb']:.4f}, "
        f"latency={row['latency_mean_ms']:.2f} ms"
    )

print(f'NEDI combined results: {nedi_combined_csv}')
print(f'NEDI summary: {summary_csv}')
print(f'AWS bicubic/NEDI comparison: {comparison_summary_csv}')
